# 02 — Business-oriented exploratory analysis

---

**Investment committee framing:** *Which observable, pre-release characteristics align with commercial outcomes — and where should we concentrate underwriting and greenlight diligence?*

This notebook treats `movies_cleaned_with_target.csv` as the **analytical film slate**: we contrast **pre-release signals** with **realized commercial outcomes** (ROI and ROI-derived success classes), articulate **capital-markets-style narratives**, export **slide-ready figures** to `plots/business_eda/` for Streamlit reuse, and record **testable hypotheses** for the modeling phase — **without** estimating models here.

**How to read each section:** scan the exhibits first, then the **Insights** block (**Observation → Business interpretation → Modeling implication**).



In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

_CWD = Path.cwd().resolve()
if (_CWD / "data" / "processed").is_dir():
    PROJECT_ROOT = _CWD
elif (_CWD.parent / "data" / "processed").is_dir():
    PROJECT_ROOT = _CWD.parent
else:
    PROJECT_ROOT = _CWD
    print("Warning: data/processed not found next to cwd or parent; using cwd.")

CLEAN_PATH = PROJECT_ROOT / "data" / "processed" / "movies_cleaned_with_target.csv"
PLOTS_DIR = PROJECT_ROOT / "plots" / "business_eda"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

FIG_W, FIG_H = 11, 5.5
plt.rcParams["figure.figsize"] = (FIG_W, FIG_H)
plt.rcParams["axes.titlesize"] = 15
plt.rcParams["axes.titleweight"] = "semibold"
plt.rcParams["axes.titlepad"] = 14
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["axes.edgecolor"] = "#333333"
plt.rcParams["axes.labelcolor"] = "#222222"
plt.rcParams["text.color"] = "#222222"
plt.rcParams["grid.alpha"] = 0.35
plt.rcParams["xtick.labelsize"] = 11
plt.rcParams["ytick.labelsize"] = 11

sns.set_theme(style="whitegrid", context="talk", font_scale=0.92)


def save_fig(name: str) -> Path:
    path = PLOTS_DIR / f"{name}.png"
    plt.savefig(path, dpi=150, bbox_inches="tight", facecolor="white", edgecolor="none")
    plt.close()
    return path


print("PROJECT_ROOT:", PROJECT_ROOT)
print("Dataset exists:", CLEAN_PATH.exists())
print("Figures will save to:", PLOTS_DIR)



In [ ]:
df = None
if not CLEAN_PATH.exists():
    print(f"Missing: {CLEAN_PATH}")
    print("Run notebooks/01_dataset_audit_and_target.ipynb first to create the cleaned CSV.")
else:
    df = pd.read_csv(CLEAN_PATH)
    if "roi" not in df.columns:
        raise ValueError("Expected column 'roi' in cleaned dataset.")
    df["log_roi"] = np.log1p(df["roi"])
    if "movie_success_class" in df.columns:
        df["movie_success_class"] = pd.Categorical(
            df["movie_success_class"],
            categories=["flop", "average", "hit"],
            ordered=True,
        )
    print("shape:", df.shape)
    display(df.head(3))
    summary_cols = [c for c in ["budget", "revenue", "roi", "log_roi", "movie_success_class"] if c in df.columns]
    display(df[summary_cols].describe(include="all"))



## 1 — ROI distribution (raw vs log)

**Business lens:** Entertainment returns are **option-like**: a long right tail of blockbusters and a left tail of write-downs. `log_roi` stabilizes the view so we can discuss **typical** economics without being blinded by a handful of extreme outcomes.



In [ ]:
if df is None:
    print("Skip: no dataframe.")
else:
    roi = df["roi"]
    fig, axes = plt.subplots(1, 2, figsize=(12, 5.2))
    axes[0].hist(roi, bins=60, color="#2c5282", edgecolor="white", linewidth=0.6, alpha=0.88)
    axes[0].set_title("ROI distribution (raw)")
    axes[0].set_xlabel("ROI (revenue / budget)")
    axes[0].set_ylabel("Count")

    axes[1].hist(df["log_roi"], bins=60, color="#1a365d", edgecolor="white", linewidth=0.6, alpha=0.88)
    axes[1].set_title("log(1 + ROI) distribution")
    axes[1].set_xlabel("log_roi")
    axes[1].set_ylabel("Count")
    plt.tight_layout()
    save_fig("01_roi_raw_and_log")

    q01, q99 = roi.quantile(0.01), roi.quantile(0.99)
    iqr = roi.quantile(0.75) - roi.quantile(0.25)
    low_iqr = roi.quantile(0.25) - 1.5 * iqr
    high_iqr = roi.quantile(0.75) + 1.5 * iqr
    extreme_high = int((roi > max(high_iqr, q99)).sum())
    extreme_low = int((roi < max(low_iqr, 0)).sum())
    print("ROI percentiles p01 / p50 / p99:", float(q01), float(roi.median()), float(q99))
    print("Approx. extreme high tail count (above ~IQR or p99):", extreme_high)
    print("Approx. extreme low tail count:", extreme_low)



### Insights — ROI distribution

- **Observation:** Raw ROI is long-right-tailed; `log_roi` compresses extremes while preserving ordering for relative comparisons.
- **Business interpretation:** A few outsized hits and deep losses dominate raw-scale moments; **median and inter-quartile** stories usually matter more than means for capital allocation conversations.
- **Modeling implication:** For any later **ROI regression**, plan for skew (transforms / robust losses); for **classification** on success bands, align thresholds explicitly to hurdle-rate language executives already use.



## 2 — Success class distribution (portfolio mix)

**Business lens:** The mix of flops, average performers, and hits sets **base rates** for precision/recall expectations and shapes how aggressively a screening tool can filter without starving the slate of upside optionality.



In [ ]:
if df is None:
    print("Skip.")
else:
    vc = df["movie_success_class"].value_counts().sort_index()
    fig, ax = plt.subplots(figsize=(8.5, 5.2))
    x = vc.index.astype(str)
    ax.bar(x, vc.values, color="#2c699a", edgecolor="white", linewidth=0.8, zorder=2)
    ax.set_title("Commercial success classes (ROI-based buckets)")
    ax.set_xlabel("movie_success_class")
    ax.set_ylabel("Count")
    ymax = float(vc.values.max())
    for i, v in enumerate(vc.values):
        ax.text(i, v + ymax * 0.015, str(int(v)), ha="center", fontsize=11, fontweight="medium")
    ax.set_ylim(0, ymax * 1.12)
    plt.tight_layout()
    save_fig("02_success_class_counts")

    props = (vc / vc.sum()).rename("share")
    display(pd.concat([vc.rename("count"), props], axis=1).round(4))



### Insights — success classes

- **Observation:** Class prevalence shows whether “hit” behaves like a **rare event** in this historical slice.
- **Business interpretation:** Rare upside events can justify selective risk-taking; the portfolio question is whether the slate is **over-concentrated** in low-upside buckets.
- **Modeling implication:** Expect **class imbalance**; prioritize **macro-F1** and per-class recall in later evaluation, not headline accuracy alone.



## 3 — Budget vs outcomes (scale vs efficiency)

**Business lens:** The central tension is **capital intensity vs capital efficiency**: larger budgets can buy distribution and spectacle, but they also raise the **revenue bar** required to clear hurdle returns.



In [ ]:
if df is None:
    print("Skip.")
else:
    sub = df[(df["budget"] > 0) & (df["revenue"] > 0)].copy()

    fig, ax = plt.subplots(figsize=(8.5, 6.2))
    ax.scatter(
        np.log1p(sub["budget"]),
        np.log1p(sub["revenue"]),
        alpha=0.18,
        s=22,
        c="#2c5282",
        edgecolors="none",
    )
    ax.set_title("Budget vs revenue (log1p scale)")
    ax.set_xlabel("log1p(budget)")
    ax.set_ylabel("log1p(revenue)")
    plt.tight_layout()
    save_fig("03a_budget_vs_revenue_log")

    fig, ax = plt.subplots(figsize=(8.5, 6.2))
    ax.scatter(
        np.log1p(sub["budget"]),
        sub["log_roi"],
        alpha=0.18,
        s=22,
        c="#1a365d",
        edgecolors="none",
    )
    ax.set_title("Budget vs log ROI")
    ax.set_xlabel("log1p(budget)")
    ax.set_ylabel("log_roi")
    plt.tight_layout()
    save_fig("03b_budget_vs_log_roi")

    sub["budget_quartile"] = pd.qcut(
        sub["budget"],
        q=4,
        labels=["Q1 — lowest spend band", "Q2 — lower-mid", "Q3 — upper-mid", "Q4 — top spend band"],
        duplicates="drop",
    )
    fig, ax = plt.subplots(figsize=(10.5, 5.8))
    sns.boxplot(
        data=sub,
        x="budget_quartile",
        y="roi",
        hue="budget_quartile",
        palette="Blues",
        dodge=False,
        linewidth=1.0,
        fliersize=2,
        ax=ax,
        legend=False,
    )
    ax.set_title("ROI by budget quartile (raw ROI — expect heavy tails)")
    ax.set_xlabel("Budget quartile")
    ax.set_ylabel("ROI")
    plt.xticks(rotation=18, ha="right")
    plt.tight_layout()
    save_fig("03c_roi_by_budget_quartile")

    fig, ax = plt.subplots(figsize=(10.5, 5.8))
    sns.violinplot(
        data=sub,
        x="budget_quartile",
        y="log_roi",
        hue="budget_quartile",
        palette="muted",
        dodge=False,
        inner="box",
        linewidth=0.9,
        ax=ax,
        legend=False,
    )
    ax.set_title("log ROI by budget quartile")
    ax.set_xlabel("Budget quartile")
    ax.set_ylabel("log_roi")
    plt.xticks(rotation=18, ha="right")
    plt.tight_layout()
    save_fig("03d_logroi_by_budget_quartile_violin")

    fig, ax = plt.subplots(figsize=(10.5, 5.8))
    sub["log_budget"] = np.log1p(sub["budget"])
    sns.boxplot(
        data=sub,
        x="movie_success_class",
        y="log_budget",
        hue="movie_success_class",
        palette="Blues",
        dodge=False,
        linewidth=1.0,
        fliersize=2,
        ax=ax,
        legend=False,
    )
    ax.set_title("log1p(budget) by success class")
    ax.set_xlabel("movie_success_class")
    ax.set_ylabel("log1p(budget)")
    plt.tight_layout()
    save_fig("03e_logbudget_by_success_class")

    med = sub.groupby("budget_quartile", observed=True)["roi"].median()
    print("Median ROI by budget quartile:")
    display(med.to_frame("median_roi").round(3))



### Insights — budget

- **Observation:** Log–log budget vs revenue typically shows **scale alignment**; budget vs ROI (or log ROI) tests whether “spend more” buys **efficiency** or mainly **absolute gross**.
- **Business interpretation:** Higher budgets often increase **gross** potential while **compressing ROI** at the margin — the classic “tentpole economics” story — but the slope should be read as associative, not causal.
- **Modeling implication:** Treat `budget` as a **primary baseline feature**; consider **non-linear** specifications and interactions with genre/window in modeling, always excluding post-release leakage.



## 4 — Genre signals (positioning & risk premia)

**Business lens:** Genre is a compact proxy for **audience positioning**, merchandising optionality, and competitive density in the release corridor — i.e., how the market prices risk *before* marketing outcomes are observed.



In [ ]:
if df is None or "main_genre" not in df.columns:
    print("Skip (needs main_genre).")
else:
    g = df.dropna(subset=["main_genre"]).copy()
    top_genres = g["main_genre"].value_counts().head(12).index
    g12 = g[g["main_genre"].isin(top_genres)]

    fig, ax = plt.subplots(figsize=(10.5, 5.5))
    order = g12["main_genre"].value_counts().index
    sns.countplot(data=g12, y="main_genre", order=order, color="#2c5282", ax=ax)
    ax.set_title("Most common primary genres (top 12)")
    ax.set_xlabel("Count")
    plt.tight_layout()
    save_fig("04a_main_genre_counts")

    min_n = 25
    vc = g["main_genre"].value_counts()
    keep = vc[vc >= min_n].index
    gf = g[g["main_genre"].isin(keep)]
    fig, ax = plt.subplots(figsize=(11.5, 6.2))
    order_gen = sorted(keep)
    sns.boxplot(
        data=gf,
        x="log_roi",
        y="main_genre",
        order=order_gen,
        hue="main_genre",
        palette="Set2",
        dodge=False,
        linewidth=0.9,
        fliersize=2,
        ax=ax,
        legend=False,
    )
    ax.set_title(f"log ROI by primary genre (n ≥ {min_n} per genre)")
    ax.set_xlabel("log_roi")
    plt.tight_layout()
    save_fig("04b_logroi_by_main_genre")

    ct = pd.crosstab(gf["main_genre"], gf["movie_success_class"], normalize="index") * 100
    fig, ax = plt.subplots(figsize=(10.5, 6.2))
    colors = ["#264653", "#2a9d8f", "#e9c46a"]
    ct.plot(kind="bar", stacked=True, ax=ax, color=colors[: ct.shape[1]], width=0.82, edgecolor="white", linewidth=0.5)
    ax.set_title("Success mix by primary genre (% within genre)")
    ax.set_xlabel("main_genre")
    ax.set_ylabel("Share (%)")
    ax.legend(title="Class", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    save_fig("04c_success_mix_by_genre")

    hit = (g["movie_success_class"] == "hit").astype(float)
    hit_rate = (
        g.assign(_hit=hit)
        .groupby("main_genre", observed=True)["_hit"]
        .mean()
        .mul(100)
        .sort_values(ascending=False)
    )
    print("Hit-rate % by main_genre (all genres; interpret thin buckets cautiously):")
    display(hit_rate.head(15).to_frame("hit_rate_pct").round(2))



### Insights — genre

- **Observation:** Frequency charts show **where the catalog concentrates**; dispersion of log ROI by genre reveals **volatility regimes** (stable cash-engine genres vs high-variance bets).
- **Business interpretation:** Some genres behave like **high-volatility assets** with asymmetric upside — useful for diversification and for calibrating **risk appetite** at greenlight.
- **Modeling implication:** Encode `main_genre` carefully; consider richer **multi-label** genre structure later (from JSON) rather than a single label alone.



## 5 — Release timing (window risk)

**Business lens:** Calendar placement is a proxy for **competitive crowding**, seasonal demand, and studio “tentpole corridor” behavior — all priced *before* opening weekend results exist in our leakage-safe framing.



In [ ]:
if df is None:
    print("Skip.")
else:
    if "release_month" not in df.columns:
        print("Missing release_month — run notebook 01 first.")
    else:
        m = df.dropna(subset=["release_month"]).copy()
        m["release_month"] = m["release_month"].astype(int)

        fig, ax = plt.subplots(figsize=(10.5, 5.5))
        order_m = list(range(1, 13))
        sns.countplot(data=m, x="release_month", order=order_m, color="#0f766e", ax=ax)
        ax.set_title("Release volume by calendar month")
        ax.set_xlabel("Month")
        ax.set_ylabel("Count")
        plt.tight_layout()
        save_fig("05a_release_month_volume")

        med_m = m.groupby("release_month", observed=True)["log_roi"].median().reindex(order_m)
        plot_m = med_m.dropna()
        fig, ax = plt.subplots(figsize=(10.5, 5.5))
        ax.bar(plot_m.index.astype(int), plot_m.values, color="#0d9488", edgecolor="white", linewidth=0.6)
        ax.set_title("Median log ROI by release month")
        ax.set_xlabel("Month")
        ax.set_ylabel("Median log_roi")
        ax.set_xticks(order_m)
        plt.tight_layout()
        save_fig("05b_median_logroi_by_month")

        if "release_quarter" in m.columns:
            fig, ax = plt.subplots(figsize=(8.5, 5.5))
            sns.boxplot(
                data=m,
                x="release_quarter",
                y="log_roi",
                hue="release_quarter",
                palette="crest",
                dodge=False,
                linewidth=0.9,
                fliersize=2,
                ax=ax,
                legend=False,
            )
            ax.set_title("log ROI by release quarter")
            ax.set_xlabel("release_quarter")
            plt.tight_layout()
            save_fig("05c_logroi_by_quarter")

        best_month = m.groupby("release_month")["log_roi"].median().sort_values(ascending=False).head(3)
        print("Top months by median log_roi:")
        display(best_month.to_frame("median_log_roi").round(4))



### Insights — release timing

- **Observation:** Release counts by month show **supply clustering**; median log ROI by month/quarter reveals whether certain windows associate with better realized economics in this archive.
- **Business interpretation:** Seasonal lifts may reflect **bigger films placed in strong windows** (selection) as much as pure calendar alpha — the right narrative is “**hypothesis for modeling controls**,” not guaranteed uplift.
- **Modeling implication:** Keep `release_month` / `release_quarter` as categorical signals; consider **era effects** and a future **time-based validation** split when you move beyond this PoC.



## 6 — Runtime (format & production scope)

**Business lens:** Runtime signals **format** (tight vs epic), production complexity, and positioning against exhibitor constraints — second-order relative to budget and genre but still decision-relevant for pacing and cost.



In [ ]:
if df is None or "runtime" not in df.columns:
    print("Skip.")
else:
    r = df.dropna(subset=["runtime"]).copy()

    fig, ax = plt.subplots(figsize=(9.5, 5.5))
    sns.histplot(r["runtime"], bins=40, kde=True, color="#475569", ax=ax, edgecolor="white", linewidth=0.4)
    ax.set_title("Runtime distribution")
    ax.set_xlabel("Runtime (minutes)")
    plt.tight_layout()
    save_fig("06a_runtime_hist")

    fig, ax = plt.subplots(figsize=(8.5, 6.2))
    samp = r.sample(min(2000, len(r)), random_state=42)
    sns.scatterplot(data=samp, x="runtime", y="log_roi", alpha=0.22, s=26, color="#1e3a5f", edgecolor=None, ax=ax)
    ax.set_title("Runtime vs log ROI (random sample ≤ 2,000 titles)")
    ax.set_xlabel("Runtime (minutes)")
    ax.set_ylabel("log_roi")
    plt.tight_layout()
    save_fig("06b_runtime_vs_logroi")

    fig, ax = plt.subplots(figsize=(10.5, 5.8))
    sns.boxplot(
        data=r,
        x="movie_success_class",
        y="runtime",
        hue="movie_success_class",
        palette="pastel",
        dodge=False,
        linewidth=0.9,
        fliersize=2,
        ax=ax,
        legend=False,
    )
    ax.set_title("Runtime by success class")
    ax.set_xlabel("movie_success_class")
    plt.tight_layout()
    save_fig("06c_runtime_by_class")



### Insights — runtime

- **Observation:** The runtime–log ROI scatter is usually **diffuse** versus budget/genre structure; class-wise boxplots can still reveal format clusters (e.g., animation/epic runtimes).
- **Business interpretation:** Runtime is rarely a **standalone** driver; it matters mainly as **context** bundled with genre and budget tier.
- **Modeling implication:** Keep runtime as a numeric baseline; only invest in non-linear transforms if validation shows incremental lift beyond trees’ default splits.



## 7 — Original language (market footprint)

**Business lens:** Language is a coarse proxy for **primary addressable market** and international vs domestic-first positioning — important for interpreting ROI benchmarks across different budget baselines.



In [ ]:
if df is None or "original_language" not in df.columns:
    print("Skip.")
else:
    lang = df.dropna(subset=["original_language"]).copy()
    top_lang = lang["original_language"].value_counts().head(12)

    fig, ax = plt.subplots(figsize=(9.5, 5.5))
    ax.barh(top_lang.index.astype(str), top_lang.values, color="#2c5282", edgecolor="white", linewidth=0.5)
    ax.invert_yaxis()
    ax.set_title("Most common original-language codes (top 12)")
    ax.set_xlabel("Count")
    plt.tight_layout()
    save_fig("07a_language_counts")

    min_n = 40
    vc = lang["original_language"].value_counts()
    keep = vc[vc >= min_n].index
    lf = lang[lang["original_language"].isin(keep)]
    fig, ax = plt.subplots(figsize=(10.5, 5.8))
    sns.boxplot(
        data=lf,
        x="original_language",
        y="log_roi",
        hue="original_language",
        palette="Set2",
        dodge=False,
        linewidth=0.9,
        fliersize=2,
        ax=ax,
        legend=False,
    )
    ax.set_title(f"log ROI by language (n ≥ {min_n} per code)")
    ax.set_xlabel("original_language")
    plt.xticks(rotation=0)
    plt.tight_layout()
    save_fig("07b_logroi_by_language")

    lang["english"] = np.where(lang["original_language"] == "en", "English", "Non-English")
    fig, ax = plt.subplots(figsize=(7.5, 5.8))
    sns.violinplot(
        data=lang,
        x="english",
        y="log_roi",
        hue="english",
        palette=["#2c5282", "#c05621"],
        dodge=False,
        inner="box",
        linewidth=0.9,
        ax=ax,
        legend=False,
    )
    ax.set_title("log ROI: English vs non-English originals")
    plt.tight_layout()
    save_fig("07c_english_vs_nonenglish_logroi")

    summ = lang.groupby("english")["log_roi"].agg(["median", "mean", "count"]).round(4)
    display(summ)



### Insights — language

- **Observation:** English-language titles often dominate the catalog, which can **mask** structural differences for non-English releases.
- **Business interpretation:** English vs non-English comparisons should be framed as **mix effects** (budget/genre composition), not cultural quality judgments.
- **Modeling implication:** Treat `original_language` as categorical with **minimum support thresholds** in plots; avoid over-interpreting thin buckets.



## 8 — Production footprint (coalition complexity)

**Business lens:** Company frequency surfaces **repeat franchise engines**; co-production counts proxy **risk-sharing** and operational complexity — both relevant to underwriting narratives pre-release.



In [ ]:
def parse_json_list(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, str):
        s = value.strip()
        if not s:
            return []
        try:
            out = json.loads(s)
            return out if isinstance(out, list) else []
        except json.JSONDecodeError:
            return []
    return []


if df is None:
    print("Skip.")
else:
    if "production_companies" in df.columns:
        names = []
        for v in df["production_companies"]:
            for d in parse_json_list(v):
                if isinstance(d, dict) and "name" in d:
                    names.append(d["name"])
        top_co = pd.Series(names).value_counts().head(15)
        fig, ax = plt.subplots(figsize=(9.5, 6.2))
        ax.barh(top_co.index.astype(str), top_co.values, color="#9b2226", edgecolor="white", linewidth=0.5)
        ax.invert_yaxis()
        ax.set_title("Top production companies (appearance count in dataset)")
        ax.set_xlabel("Appearances (titles can share banners)")
        plt.tight_layout()
        save_fig("08a_top_production_companies")
        display(top_co.to_frame("appearances"))
    else:
        print("No production_companies column in CSV.")

    for col, cap, fname in [
        ("production_company_count", 12, "08b_production_company_count"),
        ("production_country_count", 8, "08c_production_country_count"),
    ]:
        if col not in df.columns:
            print("Missing", col)
            continue
        t = df[[col, "log_roi"]].dropna().copy()
        t[col] = t[col].clip(upper=cap)
        fig, ax = plt.subplots(figsize=(11.5, 5.8))
        sns.boxplot(data=t, x=col, y="log_roi", color="#94a3b8", linewidth=0.9, fliersize=2, ax=ax)
        ax.set_title(f"log ROI vs {col} (capped at {cap} on x for readability)")
        plt.tight_layout()
        save_fig(fname)



### Insights — production structure

- **Observation:** Top banners reflect **repeat players** in the dataset; count features summarize how “multi-party” a project looks on paper.
- **Business interpretation:** More partners can mean **risk-sharing** — or coordination drag — the sign is empirical and should be stress-tested by segment.
- **Modeling implication:** Keep counts as numeric features; high-cardinality studio names need **regularized encodings** later, not raw full cardinality in a baseline.



## 9 — Correlation view (numeric drivers + outcomes)

**Business lens:** This heatmap summarizes **linear co-movement** among selected numeric fields, including **realized outcomes** (`roi`, `log_roi`) strictly for **EDA storytelling** — these outcome columns must **not** enter predictive features later.

**Reading discipline:** correlation is not causation; trees can exploit **non-linear** structure that correlations understate.



In [ ]:
if df is None:
    print("Skip.")
else:
    cols = [
        "budget",
        "runtime",
        "genre_count",
        "production_company_count",
        "production_country_count",
        "spoken_language_count",
        "roi",
        "log_roi",
    ]
    have = [c for c in cols if c in df.columns]
    miss = [c for c in cols if c not in df.columns]
    if miss:
        print("Skipping missing columns:", miss)
    cm = df[have].dropna()
    corr = cm.corr(numeric_only=True)
    fig, ax = plt.subplots(figsize=(10.5, 8.5))
    sns.heatmap(
        corr,
        annot=True,
        fmt=".2f",
        cmap="RdBu_r",
        center=0,
        ax=ax,
        square=True,
        linewidths=0.6,
        linecolor="#e2e8f0",
        annot_kws={"size": 10.5, "weight": "medium"},
        cbar_kws={"shrink": 0.82, "label": "Pearson r"},
    )
    ax.set_title("Linear correlations — numeric EDA set (outcomes included for exposition only)")
    plt.tight_layout()
    save_fig("09_correlation_heatmap")
    display(corr.round(3))



### Insights — correlations

- **Observation:** `budget` typically co-moves with scale-related outcomes; structural counts (`genre_count`, footprint counts) are usually **weaker linear** correlates but can still matter non-linearly.
- **Business interpretation:** Low correlation with ROI does not imply “not important” — it often means **thresholds and interactions** carry the signal.
- **Modeling implication:** Use this exhibit to motivate **tree ensembles** and controlled feature expansions rather than over-weighting linear intuition alone.



In [ ]:
if df is None:
    print("Run notebook 01 first to populate summary statistics.")
else:
    lines = []
    lines.append("## Auto-generated summary stats (committee appendix)")
    vc = df["movie_success_class"].value_counts(normalize=True, dropna=True).sort_index() * 100
    lines.append("Class shares (%): " + ", ".join(f"{k}: {v:.1f}" for k, v in vc.items()))
    if "release_month" in df.columns:
        bm = df.groupby(df["release_month"].dropna().astype(int))["log_roi"].median().idxmax()
        lines.append(f"Month with highest median log_roi (descriptive): {int(bm)}")
    if "main_genre" in df.columns:
        mg = df.dropna(subset=["main_genre"]).copy()
        hr = (
            mg.assign(_hit=(mg["movie_success_class"] == "hit").astype(float))
            .groupby("main_genre", observed=True)["_hit"]
            .mean()
        )
        top_hit = hr.sort_values(ascending=False).head(3)
        lines.append(
            "Top genres by hit-rate (noisy for sparse genres): "
            + ", ".join(top_hit.index.astype(str))
        )
    print("\n".join(lines))



---

## Executive read-out — for a studio investment committee

### Principal commercial findings
- **Return shape:** ROI is dominated by tail events; **bucketed success classes** translate that reality into language finance and creative leadership can act on.
- **Capital vs efficiency:** Budget scales gross potential but does not automatically buy **capital efficiency** — expect tension between tentpole scale and ROI hurdles.
- **Positioning & window:** Genre and release timing behave like **risk-factor layers** (volatility and seasonality hypotheses) that should be modeled jointly with budget, not in isolation.

### Strongest predictive hypotheses (for the next ML phase)
- **Budget tier** + **genre positioning** as the core first-order signal bundle.
- **Release window** as a categorical **competition / demand** proxy (with humility about selection effects).
- **Production footprint counts** as structural complexity / internationalization proxies.

### Risk patterns to flag in diligence
- **High-volatility genres** where median returns look acceptable but dispersion is wide — classic “option value vs loss severity” trade-offs.
- **Thin-language or thin-genre buckets** where sample noise can masquerade as insight.

### Feature engineering directions (still pre-release–aligned)
- Multi-label genre expansion; **studio / banner** frequency tiers with regularization; decade buckets from `release_year` if retained without leakage; optional inflation adjustment *only* with external indices and clear documentation.

### Limitations & caution
- **TMDB archive ≠ live greenlight data** (marketing spend, competitive calendar, talent deals largely unobserved).
- **Associative patterns only** — use outputs to **steer questions and triage**, not to replace creative, distribution, and marketing judgment.

### How this informs the next ML phase
- Proceed to **leakage-safe classification** on `movie_success_class` with **macro-F1** emphasis, stratified evaluation, and **probability calibration** if the product story requires interpretable risk bands.

---

**Deliverables:** figures under `plots/business_eda/` (`01_` … `09_` prefixes) are sized for reuse in **Streamlit** and slide decks.

